# Data exploration (task 3, owner A)

Three counts that inform the label design:
1. How many businesses are restaurants
2. Open vs closed split among restaurants (`is_open`)
3. Reviews per year, all businesses and restaurants only, to confirm there is enough volume either side of the chosen cutoff

Also: the year of each closed restaurant's last review, since that is how we infer closure timing.

The review file is read in chunks via `iter_reviews()` from `src/data.py`; the full pass takes several minutes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_business, load_restaurants, iter_reviews

biz = load_business()
biz.shape  # expected (150346, 14)

## 1. Restaurant count
Uses the shared rule in `src/data.py`: a business counts as a restaurant if "Restaurants" is one of its categories (exact match after splitting on commas, so e.g. "Restaurant Supplies" is excluded).

In [ ]:
rest = load_restaurants()  # restaurant rule lives in src/data.py
print(f"Restaurants: {len(rest):,} of {len(biz):,} businesses ({len(rest) / len(biz):.1%})")

## 2. Open vs closed

In [ ]:
split = rest["is_open"].map({1: "open", 0: "closed"}).value_counts()
pd.DataFrame({"restaurants": split, "share": (split / split.sum()).round(3)})

In [ ]:
# Closed share by state, for states with at least 500 restaurants
by_state = rest.groupby("state").agg(restaurants=("is_open", "size"), closed_share=("is_open", lambda s: 1 - s.mean()))
by_state[by_state["restaurants"] >= 500].sort_values("restaurants", ascending=False).round(3)

## 3. Reviews per year
One pass over the review file. Per chunk we keep only `business_id` and `date` (already parsed as dates by the loader), tally reviews per year, and track each restaurant's last review date.

In [ ]:
rest_ids = set(rest["business_id"])
all_years, rest_years, last_review = [], [], []

for chunk in iter_reviews(columns=["business_id", "date"]):
    all_years.append(chunk["date"].dt.year.value_counts())
    r = chunk[chunk["business_id"].isin(rest_ids)]
    rest_years.append(r["date"].dt.year.value_counts())
    last_review.append(r.groupby("business_id")["date"].max())

per_year = pd.DataFrame({
    "all_reviews": pd.concat(all_years).groupby(level=0).sum(),
    "restaurant_reviews": pd.concat(rest_years).groupby(level=0).sum(),
}).fillna(0).astype(int).sort_index()
per_year.index.name = "year"
last_review = pd.concat(last_review).groupby(level=0).max()

print(f"Total reviews: {per_year['all_reviews'].sum():,}  (expected ~7.0M)")
per_year

In [ ]:
ax = per_year.plot(kind="bar", figsize=(10, 4))
ax.set_ylabel("Reviews")
ax.set_title("Yelp reviews per year")
plt.tight_layout()

## 4. Year of last review, closed vs open restaurants
Closed restaurants have no closure date, so the last review stands in for it. This shows how many closures each year could supply as positive labels.

In [ ]:
rest_last = rest.set_index("business_id")[["is_open"]].join(last_review.rename("last_review"))
print(f"Restaurants with no reviews in the review file: {rest_last['last_review'].isna().sum():,}")
last_year = (
    rest_last.dropna()
    .assign(status=lambda d: d["is_open"].map({1: "open", 0: "closed"}), year=lambda d: d["last_review"].dt.year)
    .pivot_table(index="year", columns="status", values="is_open", aggfunc="size", fill_value=0)
)
last_year